In [ ]:
# 第9周-Day4：ReleaseChannel / TrafficPolicy — 为什么需要灰度？# matplotlib 中文字体配置from matplotlib import font_managerimport matplotlib.pyplot as pltfont_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"font_manager.fontManager.addfont(font_path)font_name = font_manager.FontProperties(fname=font_path).get_name()plt.rcParams["font.family"] = font_nameplt.rcParams["axes.unicode_minus"] = Falseprint("使用字体:", font_name)

## 📅 Week 9 - Day 4 | 2026-07-30### ReleaseChannel / TrafficPolicy — 为什么需要灰度？| 项目 | 内容 ||---|---|| **本周主题** | Domain Deep Dive — 逐一拆解 LangChat 核心对象 || **今日主题** | ReleaseChannel（晋升指针）与 TrafficPolicy（流量策略）|| **链路位置** | Supply Chain 末端 → Runtime 始端（制品晋升与流量路由的交界）|| **前置知识** | Day1 BlueprintVersion、Day2 SkillRelease、Day3 Deployment |

## ❓ 今日核心问题### 为什么需要灰度？不能一次全量？Jason 做了 26 年 ERP。你一定遇到过这种场景：一个财务模块升级，周一早上全量推上线，结果某个边缘税率计算逻辑在新版本里出错了，全公司当天没法做账。你当时一定想过：**"如果能先让 5% 的人用新版本，确认没问题再全推就好了。"**这就是灰度发布（Canary Release）的本质——**不是技术炫技，而是风险控制的工程纪律。**LangChat 的回答更进一层：灰度不是部署工具的附属功能，而是**架构层面两个独立对象**——`ReleaseChannel`（晋升指针）和 `TrafficPolicy`（流量策略）——各司其职，确保灰度可控、可审计、可回滚。

## 🗣 人话解释（Jason 26年 ERP 经验）**传统 ERP 的"发布"**：开发 → 测试 → 上线。上线就是全量替换。出了问题？回滚到昨天的备份。这个过程里，"哪个版本在生产"、"谁批的"、"流量怎么切"全混在一个动作里。**LangChat 的"发布"拆成三个独立动作**：| 动作 | 对象 | 做什么 | ERP 类比 ||---|---|---|---|| 晋升（Promotion）| ReleaseChannel | 把 scope 内的指针指向新版本 digest | "标记 v2.1 为当前正式版本" || 物化（Materialization）| DeploymentRevision | 把 SkillRelease digest + 环境 = 完整运行时闭包 | "在生产环境安装 v2.1" || 切流（Traffic Routing）| TrafficPolicy | 决定多少流量到新版本、多少到旧版本 | "先让 5% 的用户用 v2.1" |三个动作独立。你可以晋升了但还没部署。你也可以部署了但只给 1% 流量。**这就是灰度存在的理由：把"哪个版本是正式版"和"生产实际在跑哪个版本"解耦。**

In [ ]:
# LangChat 架构位置：Supply Chain 末端与 Runtime 始端的交界fig, ax = plt.subplots(figsize=(14, 8))ax.axis('off')ax.set_xlim(0, 10)ax.set_ylim(0, 10)# Titleax.text(5, 9.5, 'Supply Chain → Runtime：ReleaseChannel 与 TrafficPolicy 的交界',         ha='center', fontsize=14, fontweight='bold', color='#333')# Supply Chain areafrom matplotlib.patches import FancyBboxPatchsc_box = FancyBboxPatch((0.5, 5.5), 4, 3, boxstyle='round,pad=0.3',                          facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2)ax.add_patch(sc_box)ax.text(2.5, 8.2, 'Supply Chain 层', ha='center', fontsize=12, fontweight='bold', color='#1565C0')ax.text(2.5, 7.5, 'BlueprintVersion', ha='center', fontsize=9)ax.text(2.5, 7.0, 'SkillRelease (制品)', ha='center', fontsize=9)ax.text(2.5, 6.3, '★ ReleaseChannel (晋升指针)', ha='center', fontsize=10, fontweight='bold', color='#E91E63')ax.text(2.5, 5.8, 'PromotionEvent (审计)', ha='center', fontsize=8)# Runtime areart_box = FancyBboxPatch((5.5, 5.5), 4, 3, boxstyle='round,pad=0.3',                          facecolor='#FFF3E0', edgecolor='#E65100', linewidth=2)ax.add_patch(rt_box)ax.text(7.5, 8.2, 'Runtime 层', ha='center', fontsize=12, fontweight='bold', color='#E65100')ax.text(7.5, 7.5, 'Deployment', ha='center', fontsize=9)ax.text(7.5, 7.0, 'DeploymentRevision (闭包)', ha='center', fontsize=9)ax.text(7.5, 6.3, '★ TrafficPolicy (流量策略)', ha='center', fontsize=10, fontweight='bold', color='#E91E63')ax.text(7.5, 5.8, 'Execution', ha='center', fontsize=8)# Arrow betweenax.annotate('', xy=(5.5, 6.5), xytext=(4.5, 6.5),            arrowprops=dict(arrowstyle='->', lw=2.5, color='#4CAF50'))ax.text(5, 6.8, '部署操作', ha='center', fontsize=8, color='#4CAF50', fontweight='bold')ax.text(5, 6.2, '(一次性解析)', ha='center', fontsize=7, color='#4CAF50')# Business Domainbd_box = FancyBboxPatch((2, 1), 6, 3.5, boxstyle='round,pad=0.3',                          facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=2)ax.add_patch(bd_box)ax.text(5, 4.2, 'Business Domain Layer', ha='center', fontsize=12, fontweight='bold', color='#2E7D32')ax.text(5, 3.5, 'DigitalEmployeeDefinition', ha='center', fontsize=10)ax.text(5, 2.8, 'CapabilityRegistry', ha='center', fontsize=9)ax.text(5, 2.1, 'EffectRegistry', ha='center', fontsize=9)# Connectionsax.annotate('', xy=(5, 5.5), xytext=(5, 4.5),            arrowprops=dict(arrowstyle='->', lw=1.5, color='#666', linestyle='dashed'))plt.title('Day4: ReleaseChannel 与 TrafficPolicy 的架构位置', fontsize=13, pad=15)plt.tight_layout()plt.show()

## 📋 ADR 依据### Domain Model SC-14 ReleaseChannel（§7.10）核心不变量：1. **单点性**：`(scope, channel_name)` 在任一时刻只指向一个精确 SkillRelease digest（或空）2. **晋升受审计**：每次移动必产出 `PromotionEvent`3. **不影响服务流量**：Channel 移动不直接改变 TrafficPolicy 或 DeploymentRevision 的活跃性显式禁止：- ❌ 不在运行时请求路径中- ❌ 不指向 DeploymentRevision- ❌ 不路由流量- ❌ Channel 移动不改变已 Serving 的 Deployment 或 TrafficPolicy### Domain Model RT-03 TrafficPolicy（§8.2）核心不变量：1. **精确引用**：所有路由目标必须是具体 DeploymentRevision ID + digest2. **确定性 cohort 路由**：相同稳定分流键永远划入同一 cohort3. **版本演进**：变更必生成新版本，不原地修改显式禁止：- ❌ 不读 Catalog- ❌ 不读 ReleaseChannel 作为路由依据- ❌ 不路由到 "latest"- ❌ 不路由到 mutable name### 跨对象不变量（Domain Model §10.4）- **§10.4-4** Channel 与流量解耦：ReleaseChannel 移动不改变 TrafficPolicy 或已 Serving 的 DeploymentRevision- **§10.4-11** ReleaseChannel 归属固定：严格属 Supply Chain，不得在运行时请求路径中引用

In [ ]:
# ReleaseChannel 单点性可视化fig, axes = plt.subplots(1, 2, figsize=(14, 6))# Left: ReleaseChannel 单点映射ax1 = axes[0]ax1.axis('off')ax1.set_title('ReleaseChannel：scope 内单点指针', fontsize=12, fontweight='bold')# Channel scope boxfrom matplotlib.patches import FancyBboxPatchch_box = FancyBboxPatch((1, 3), 3, 3, boxstyle='round,pad=0.2',                         facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2)ax1.add_patch(ch_box)ax1.text(2.5, 5.5, 'Channel', ha='center', fontsize=11, fontweight='bold')ax1.text(2.5, 5.0, 'scope: (tenant, ws, env, "prod")', ha='center', fontsize=7)ax1.text(2.5, 4.3, 'digest: sha256:aaa...', ha='center', fontsize=9, color='#E91E63', fontweight='bold')ax1.text(2.5, 3.5, '(只能指向一个)', ha='center', fontsize=8, color='#666')# Digestsax1.text(5.5, 5.5, 'sha256:aaa...', fontsize=9, color='#4CAF50')ax1.text(5.5, 4.5, 'sha256:bbb...', fontsize=9, color='#999')ax1.text(5.5, 3.5, 'sha256:ccc...', fontsize=9, color='#999')ax1.annotate('✅ 当前', xy=(4.8, 5.5), xytext=(4, 4.3),             arrowprops=dict(arrowstyle='->', color='#4CAF50', lw=2))ax1.text(6.5, 5.5, '(promote 替换)', fontsize=7, color='#4CAF50')ax1.text(6.5, 4.5, '(被替换)', fontsize=7, color='#999')ax1.text(6.5, 3.5, '(未使用)', fontsize=7, color='#999')# Right: TrafficPolicy cohort routingax2 = axes[1]ax2.axis('off')ax2.set_title('TrafficPolicy：按 cohort 分流', fontsize=12, fontweight='bold')# Traffic Policytp_box = FancyBboxPatch((0.5, 3), 3, 3, boxstyle='round,pad=0.2',                         facecolor='#FFF3E0', edgecolor='#E65100', linewidth=2)ax2.add_patch(tp_box)ax2.text(2, 5.5, 'TrafficPolicy v2', ha='center', fontsize=11, fontweight='bold')ax2.text(2, 5.0, 'cohort hash(tenant_id)', ha='center', fontsize=7)ax2.text(2, 4.3, '95% → rev-41', ha='center', fontsize=9, color='#1565C0')ax2.text(2, 3.5, '5% → rev-42', ha='center', fontsize=9, color='#E91E63')# Revisionsrev1 = FancyBboxPatch((4.5, 4.8), 2.5, 1.2, boxstyle='round,pad=0.15',                        facecolor='#BBDEFB', edgecolor='#1565C0', linewidth=1.5)ax2.add_patch(rev1)ax2.text(5.75, 5.6, 'DeploymentRevision', ha='center', fontsize=8, fontweight='bold')ax2.text(5.75, 5.1, 'rev-41 (旧版)', ha='center', fontsize=8)rev2 = FancyBboxPatch((4.5, 2.8), 2.5, 1.2, boxstyle='round,pad=0.15',                        facecolor='#FFCDD2', edgecolor='#E91E63', linewidth=1.5)ax2.add_patch(rev2)ax2.text(5.75, 3.6, 'DeploymentRevision', ha='center', fontsize=8, fontweight='bold')ax2.text(5.75, 3.1, 'rev-42 (灰度新版)', ha='center', fontsize=8)ax2.annotate('95%', xy=(4.5, 5.2), xytext=(3.5, 4.8),             arrowprops=dict(arrowstyle='->', color='#1565C0', lw=2))ax2.annotate('5%', xy=(4.5, 3.4), xytext=(3.5, 3.8),             arrowprops=dict(arrowstyle='->', color='#E91E63', lw=2))plt.tight_layout()plt.show()

## 🔍 代码验证（/root/langchat）### ReleaseChannel 实现```python@dataclass(frozen=True)class ChannelScope:    # (tenant, workspace, environment, channel_name) = 唯一定位class ReleaseChannel:    def promote(self, scope: ChannelScope, digest: str, operator, promoted_at):        # 单点映射：scope 内只能指向一个 digest        # 产出 PromotionEvent 审计记录    def get(self, scope: ChannelScope) -> str:        # 返回当前 digest 或空字符串    def unpin(self, scope, operator, unpinned_at):        # 清除指针，也产出审计事件```### 单元测试验证```pythondef test_repin_replaces_digest():    # promote 两次，只保留最新 digest ✅def test_different_scopes_are_independent():    # 两个 scope 的 channel 互不干扰 ✅```### TrafficPolicy 实现```python@dataclass(frozen=True)class TrafficPolicy:    policy_id: str    revision_id: str          # 必须是具体 revision ID    revision_digest: str      # 必须是精确 digest    percentage: int           # 0-100```### 关键测试：拒绝一切非精确引用- `test_latest_rejected` → `"latest"` 被拒绝 ✅- `test_channel_name_rejected` → Channel 名被拒绝 ✅- `test_concrete_revision_accepted` → 只接受具体 revision ✅**代码忠实实现了 ADR 设计：**1. ReleaseChannel 是纯控制面指针，不碰流量2. TrafficPolicy 在构造时就拒绝一切非精确引用3. 部署管道严格遵循 Channel→Digest→Revision→Register 四步分离

In [ ]:
# LangChat → MI CRE（商业地产运营）场景映射fig, ax = plt.subplots(figsize=(14, 6))ax.axis('off')data = [    ['ReleaseChannel', '"合同审核机器人版本标记"', '标记 v2.1 为正式版，但生产可能还在跑 v2.0'],    ['PromotionEvent', '"版本变更审批单"', '谁批的、什么时候、从哪个版本到哪个版本——全留痕'],    ['DeploymentRevision', '"实际部署的机器人实例"', 'v2.1 机器人 + ERP 接口 + 知识库快照'],    ['TrafficPolicy', '"租户分流规则"', '先让 3 个租户用新版，其他 47 个继续旧版'],    ['Cohort Hash', '"租户级粘性路由"', '租户 A 今天用新版，明天还是新版'],]col_labels = ['LangChat 概念', 'MI CRE 场景对应', '解释']y = 0.95ax.text(0.5, y + 0.04, 'MI 购物中心 50 个租户：新版合同审核 SkillRelease 灰度上线',         ha='center', fontsize=12, fontweight='bold', transform=ax.transAxes)# Headerfor j, label in enumerate(col_labels):    x = [0.02, 0.28, 0.52][j]    w = [0.24, 0.22, 0.46][j]    ax.add_patch(plt.Rectangle((x, y - 0.06), w, 0.05, transform=ax.transAxes,                                 facecolor='#37474F', alpha=0.9))    ax.text(x + w/2, y - 0.035, label, ha='center', va='center', fontsize=9,             color='white', fontweight='bold', transform=ax.transAxes)# Rowsfor i, row in enumerate(data):    ry = y - 0.06 - (i+1) * 0.075    bg = '#E3F2FD' if i % 2 == 0 else '#FFF3E0'    for j, val in enumerate(row):        x = [0.02, 0.28, 0.52][j]        w = [0.24, 0.22, 0.46][j]        ax.add_patch(plt.Rectangle((x, ry), w, 0.07, transform=ax.transAxes,                                     facecolor=bg, alpha=0.7, edgecolor='#CCC'))        color = '#E91E63' if j == 0 else '#333'        fw = 'bold' if j == 0 else 'normal'        ax.text(x + 0.01, ry + 0.035, val, ha='left', va='center', fontsize=8,                color=color, fontweight=fw, transform=ax.transAxes)ax.set_xlim(0, 1)ax.set_ylim(0.2, 1)plt.tight_layout()plt.show()

In [ ]:
# 三种灰度方案对比fig, ax = plt.subplots(figsize=(14, 7))ax.axis('off')dimensions = ['版本标记\n与流量', '灰度粒度', '回滚方式', '审计追踪', '版本精确性', '运行时安全']plan_a = ['混合', '无', '恢复备份', '日志', '"v2.1"', '依赖人工']plan_b = ['部分分离', '按服务器', '切回旧实例', '部署记录', 'image tag', '依赖CI/CD']plan_c = ['完全分离', '按租户/cohort', '新建TP版本', 'Promotion+TP链', 'content digest', '架构层强制']x = np.arange(len(dimensions))width = 0.25# Score: 1=poor, 2=ok, 3=excellentscores_a = [1, 1, 1, 1, 1, 1]scores_b = [2, 2, 2, 2, 2, 2]scores_c = [3, 3, 3, 3, 3, 3]colors = ['#EF5350', '#FFA726', '#66BB6A']bars1 = ax.bar(x - width, scores_a, width, label='传统ERP（直接替换）', color=colors[0])bars2 = ax.bar(x, scores_b, width, label='DevOps（蓝绿部署）', color=colors[1])bars3 = ax.bar(x + width, scores_c, width, label='LangChat（Channel+TP）', color=colors[2])ax.set_xticks(x)ax.set_xticklabels(dimensions, fontsize=9)ax.set_yticks([1, 2, 3])ax.set_yticklabels(['弱', '一般', '强'], fontsize=9)ax.set_ylim(0, 4)ax.set_title('灰度发布三种方案对比：LangChat 在每个维度都最强', fontsize=13, fontweight='bold')ax.legend(loc='upper left', fontsize=10)# Add value labelsfor bars, vals_a, vals_b, vals_c in [(bars1, plan_a, None, None), (bars2, None, plan_b, None), (bars3, None, None, plan_c)]:    pass# Add text labels on barsfor i, (a, b, c) in enumerate(zip(plan_a, plan_b, plan_c)):    ax.text(x[i] - width, scores_a[i] + 0.1, a, ha='center', fontsize=6, rotation=0)    ax.text(x[i], scores_b[i] + 0.1, b, ha='center', fontsize=6, rotation=0)    ax.text(x[i] + width, scores_c[i] + 0.1, c, ha='center', fontsize=6, rotation=0)import numpy as npplt.tight_layout()plt.show()

## 🧠 架构师思考题**场景**：MI 集团有 3 个业态（购物中心、写字楼、酒店），每个业态有独立的租户群体。合同审核机器人发布了 v3.0，新增了按业态差异化的条款审核逻辑。**问题**：1. 你会设计几个 ReleaseChannel？Channel scope 怎么定义？2. 三个业态的灰度策略应该相同还是不同？3. 如果写字楼的 v3.0 出了问题需要回滚，购物中心和酒店受不受影响？4. 回滚时，ReleaseChannel 指针要动吗？TrafficPolicy 怎么变？**参考思路**：- Channel scope 可能是 `(tenant_group, industry, environment, channel_name)`- 不同业态的风险承受力不同 → 灰度比例可以不同- 如果 Channel 按业态分 → 回滚一个业态不影响其他- 回滚 = 新建 TrafficPolicy 版本指向旧 Revision → Channel 不一定要动

## 💡 我的理解变化**以前以为**：灰度发布就是部署工具的一个功能——在 CI/CD 流水线里配个百分比就行了。**现在知道**：LangChat 把灰度拆成了**两个独立架构对象**：- ReleaseChannel 是"版本标记"（Supply Chain 的事）- TrafficPolicy 是"流量切分"（Runtime 的事）两者在架构上完全解耦，在跨对象不变量中被显式约束（§10.4-4 和 §10.4-11）。**更深一层的认知**：灰度的核心不是"能不能按比例切流量"，而是**"版本标记与实际运行状态的解耦程度"**。传统方案的问题是：标记了正式版 = 全量上线，没有中间态。LangChat 的设计让你可以标记了正式版但只给 1% 流量——这个"标记但不全量"的中间态，才是灰度发布的工程价值。

## 📖 术语表| 英文术语 | 音标 | 释义 ||---|---|---|| ReleaseChannel | /rɪˈliːs ˈtʃænl/ | 晋升指针，scope 内单点指向当前正式版 digest || TrafficPolicy | /ˈtræfɪk ˈpɒlɪsi/ | 流量策略，按 cohort 分流到不同 DeploymentRevision || PromotionEvent | /prəˈməʊʃən ɪˈvent/ | 晋升事件，每次 Channel 移动的审计记录 || Cohort | /ˈkəʊhɔːt/ | 分流队列，相同分流键的稳定分组 || Canary Release | /kəˈneəri rɪˈliːs/ | 金丝雀发布（灰度发布），先小流量验证再全量 || Materialization | /məˌtɪəriəlaɪˈzeɪʃən/ | 物化：从 digest + 环境到完整运行时闭包 |

## ✏️ 课堂练习**Q1**: ReleaseChannel 的 promote 操作会改变 TrafficPolicy 吗？> 不会。§10.4-4 明确约束：Channel 移动不改变 TrafficPolicy 或已 Serving 的 DeploymentRevision。两者完全解耦。**Q2**: TrafficPolicy 能路由到 `"latest"` 或 Channel 名吗？> 不能。RT-03 显式禁止：所有路由目标必须是具体 DeploymentRevision ID + digest。构造时就 reject。**Q3**: 为什么说回滚是"前进"而不是"后退"？> LangChat 中回滚 = 新建一个 TrafficPolicy 版本，指向旧 DeploymentRevision。旧 Revision 本身不变（immutable），只是流量策略变了。所有操作都是前向的、可审计的。

## 🔗 明日连接**Day5：DigitalEmployeeDefinition — 为什么数字员工不拥有 Runtime？**这是 Week 9 最后一个对象。前面四天看了 Release/Deployment 的制品和流量侧，明天看"产品语义锚点"——数字员工的定义本身。**Semantic Layer 定位**：```Ontology（存在什么）  └─ Domain Model（对象边界）       ├─ ReleaseChannel = "哪个版本是正式版"（Supply Chain 指针）       ├─ TrafficPolicy = "生产实际跑哪个版本"（Runtime 流量策略）       └─ DigitalEmployeeDefinition = "这个 AI 应用是什么"（Business Domain 语义锚）```---*📝 Week 9 Day 4 · 灰度不是功能，是架构纪律*